[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Carlo-Broschi/statistics-python-for-students/blob/main/06_%E7%99%BA%E5%B1%95_%E3%83%9E%E3%83%BC%E3%82%B1%E5%88%86%E6%9E%90/01_%E3%83%89%E3%83%A9%E3%82%A4%E3%83%90%E3%83%BC%E5%88%86%E6%9E%90_%E5%9B%9E%E5%B8%B0.ipynb)

# 発展マーケ-01. ドライバー分析（重回帰・ロジスティック回帰）

> 🟢 Colab（ブラウザ）で実行できます。**最初に「① セットアップ」セルを実行**してください。
> 📌 **発展編（統計検定2級の先：準1級〜実務/データサイエンス）**。scikit-learn・statsmodels を使います（Colabは導入済み。ローカルは `uv add scikit-learn statsmodels`）。

「**何が受注を左右しているか**」を回帰で定量化します。
受注は 0/1 なので、確率を扱える **ロジスティック回帰** が主役です。

In [ ]:
# === ① セットアップ（最初に実行してください）===
import pandas as pd               # 表データ
import numpy as np                # 数値計算
import matplotlib.pyplot as plt   # グラフ描画
import os
plt.rcParams['axes.unicode_minus'] = False   # マイナス記号の文字化け防止
# ローカルに無ければ公開リポジトリ(raw)から読み込む共通関数
RAW = 'https://raw.githubusercontent.com/Carlo-Broschi/statistics-python-for-students/main/data/'
def load(name):
    p = f'../data/{name}'
    return pd.read_csv(p) if os.path.exists(p) else pd.read_csv(RAW + name)
btob = load('sales_btob.csv')   # BtoB商談データ
btob.head()

### 📋 使うデータ：`sales_btob.csv`（架空のBtoB商談400件）
1行＝商談1件。`受注`は 1=成約 / 0=失注。

| 商談ID | 受注日 | 業界 | 獲得チャネル | 商談金額 | 担当者 | 受注 |
|---|---|---|---|---|---|---|
| D0001 | 2026-01-15 | 小売 | 紹介 | 1,211,000 | 佐藤 | 0 |
| D0002 | 2026-02-05 | 医療 | テレアポ | 400,000 | 田中 | 0 |
| D0003 | 2026-02-19 | IT | 紹介 | 542,000 | 高橋 | 1 |

## 🎯 この章でできるようになること
- 重回帰とロジスティック回帰を使い分けられる
- オッズ比で「受注を左右する要因」を読める
- 受注確率を予測して施策に使える

**前提**：統計2級06（重回帰）　/　**所要**：約35分　/　**レベル**：発展（準1級〜実務）

## 1. ウォームアップ：重回帰で「商談金額」を業界で説明

質的変数（業界）はダミー変数にして投入します（`C(業界)`）。

In [ ]:
import statsmodels.formula.api as smf            # 回帰
m = smf.ols('商談金額 ~ C(業界)', data=btob).fit()  # 業界で商談金額を説明
print('決定係数 R^2:', round(m.rsquared, 3))
print(m.params.round(0))   # 基準業界との金額差（偏回帰係数）

## 2. ロジスティック回帰：受注(0/1)の要因

受注は2値なので直線回帰は不適。**ロジスティック回帰**は「受注する確率」を0〜1で扱えます。

> 📐 **数IIIメモ**：ロジスティック回帰は内部で **log（対数）と eˣ** を使い、結果を0〜1の確率に変換します。導出は大学範囲なので不要。**『オッズ比＝何倍そうなりやすいか』**の読み方だけ押さえればOK。

In [ ]:
logit = smf.logit('受注 ~ 商談金額 + C(獲得チャネル)', data=btob).fit()  # 受注の要因を推定
print('pseudo R^2:', round(logit.prsquared, 3))
print(logit.params.round(3))   # 係数（プラスほど受注しやすい）

## 3. オッズ比で読む（係数を直感的に）

係数を `exp()` すると **オッズ比**。「基準チャネル(Web広告)に比べ受注オッズが何倍か」を表します。

In [ ]:
odds = np.exp(logit.params).round(2)   # 係数→オッズ比
print('オッズ比（基準 = Web広告）:')
print(odds)

**読み取り**：`紹介` ≈ 5倍、`展示会` ≈ 4.5倍 受注しやすい（基準=Web広告）。
`商談金額` のオッズ比はほぼ1＝金額の大小は受注に効いていない。→ **チャネルが受注のドライバー**。

## 4. 受注確率を予測する

同じ金額でもチャネルで受注確率が変わることを確認します。

In [ ]:
# 金額は同じ100万円、チャネルだけ変えて受注確率を予測
new = pd.DataFrame({'商談金額':[1000000, 1000000], '獲得チャネル':['紹介', 'Web広告']})
new['受注確率'] = logit.predict(new).round(3)   # モデルで確率を予測
print(new)

→ 「紹介」を増やす・「Web広告」のクオリティを上げる、といった施策の根拠になります。

## 🧭 まとめ
- 受注の主ドライバーは **獲得チャネル**（紹介・展示会が強い）。金額はほぼ無関係。
- ロジスティック回帰なら「受注確率」を予測でき、施策の効果を見積もれる。

> 重回帰（直線）は連続値の予測、ロジスティック回帰は0/1（確率）の予測、と使い分けます。

> ⚠️ **よくある間違い**：受注は0/1なので**直線回帰は不適**（確率が0〜1をはみ出す）。→ ロジスティック回帰。また回帰で関係が出ても、無作為化した実験でない限り**因果**とは言い切れない。

## 🧠 確認テスト（自動採点）

`ans = None` を**自分の計算で置き換えて実行**すると、その場で正誤が出ます（答えは表示されません）。

In [ ]:
# 採点用の関数を実行（答え合わせに使うだけ）
def _check(name, got, want, tol=1e-2):
    if got is None:
        print(f'⬜ {name}: まだ入力されていません（ans を埋めてね）'); return
    ok = abs(got - want) <= tol
    print(('✅ 正解！ ' if ok else '❌ もう一度 ') + f'{name}: あなたの答え = {got}')

In [ ]:
import numpy as np
# Q1: ロジスティック回帰の係数が 0.7 のとき、オッズ比 exp(係数) を ans に
ans = None   # 例: np.exp(0.7)
_check('Q1 オッズ比', ans, np.exp(0.7))

In [ ]:
import numpy as np
# Q2: ロジスティックの予測確率 1/(1+e^-z)。z=0 のとき ans は？
ans = None   # 例: 1/(1+np.exp(0))
_check('Q2 予測確率(z=0)', ans, 1/(1+np.exp(0)))

---
## 🏆 練習問題

**問1.** `商談金額 ~ C(獲得チャネル)` の重回帰で、チャネル別の平均単価の差を見よう。

**問2.** ロジスティック回帰に `C(業界)` も加え、pseudo R² が上がるか比べよう。

**問3.** `テレアポ` の受注オッズ比はいくつ？ Web広告より受注しやすい？

**問4.** 担当者別の受注率を `groupby` で出し、回帰の結果と照らし合わせよう。

In [ ]:
# 問1〜4


> 🔑 **解答例は別ページ**（ネタバレ防止）👉 **[解答例を開く](https://github.com/Carlo-Broschi/statistics-python-for-students/blob/main/%E8%A7%A3%E7%AD%94%E9%9B%86/06_%E7%99%BA%E5%B1%95_%E3%83%9E%E3%83%BC%E3%82%B1%E5%88%86%E6%9E%90/01_%E3%83%89%E3%83%A9%E3%82%A4%E3%83%90%E3%83%BC%E5%88%86%E6%9E%90_%E5%9B%9E%E5%B8%B0.md)**

## 📒 用語集 ＆ チートシート

| 用語 | 意味 |
|---|---|
| 重回帰 | 連続値を複数要因で予測 |
| ロジスティック回帰 | 0/1（確率）を予測 |
| オッズ | 起こる/起こらない の比 |
| オッズ比 | exp(係数)。基準比 何倍なりやすいか |
| 偏回帰係数 | 他を一定にしたときの効き目 |

In [ ]:
# チートシート（回帰・ロジスティック）
import statsmodels.formula.api as smf
import numpy as np
m = smf.logit('受注 ~ 商談金額 + C(獲得チャネル)', data=btob).fit(disp=0)
print(np.exp(m.params).round(2))   # オッズ比（基準=Web広告）